# P8-2 · MLOps

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q wandb

## Lý thuyết

In [ ]:
import wandb, math
def train():                                   # thay cho vòng train thật: trả về (bước, loss)
    for step in range(100):
        yield step, 2.0 * math.exp(-step / 30) + 0.1
f1 = 0.83

# mode="offline": chạy không cần tài khoản, log lưu ở thư mục wandb/; bỏ đi để đẩy lên trang W&B
run = wandb.init(project="lab08-extract", mode="offline", config={"lr": 2e-4, "r": 16, "epochs": 3, "data": "v3"})
for step, loss in train():
    wandb.log({"train/loss": loss}, step=step)
wandb.log({"test/f1": f1})
run.finish()
import os
print("đã ghi log vào", os.path.relpath(run.dir))

đã ghi log vào wandb/offline-run-20261106_183028-5vb4i7fw/files


In [ ]:
import numpy as np
def gate(scores_new, scores_base, max_drop=0.0):
    d = np.asarray(scores_new, float) - np.asarray(scores_base, float)   # chênh lệch từng mẫu, cùng thứ tự
    mean, se = d.mean(), d.std(ddof=1) / np.sqrt(len(d))
    upper = mean + 1.96 * se                                            # cận trên khoảng tin cậy 95%
    return ("CHẶN" if upper < -max_drop else "QUA"), round(mean, 4), round(upper, 4)

# thử: 300 mẫu, model mới sai thêm 30 mẫu so với baseline
rng = np.random.default_rng(0)
base = (rng.random(300) < 0.8).astype(float)
new = base.copy(); new[:30] = 0
print(gate(new, base))                 # tụt thật: chặn
print(gate(base, base))                # như nhau: qua

('CHẶN', -0.0733, -0.0438)
('QUA', 0.0, 0.0)


## Bài tập 4: Workflow CI có cổng eval {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
%%writefile eval_gate.py
# eval_gate.py: chạy bộ eval nhỏ, so với baseline đã lưu, thoát mã 1 nếu tụt quá ngưỡng
import json, sys

def main(results_path="eval_results.json", baseline_path="baseline.json", max_drop=0.01):
    cur = json.load(open(results_path))
    base = json.load(open(baseline_path))
    failed = []
    for metric, base_val in base.items():
        drop = base_val - cur.get(metric, 0)
        status = "OK" if drop <= max_drop else "TỤT"
        print(f"{metric:12s} baseline {base_val:.3f} | mới {cur.get(metric, 0):.3f} | {status}")
        if drop > max_drop:
            failed.append(metric)
    sys.exit(1 if failed else 0)

if __name__ == "__main__":
    main()

Writing eval_gate.py


In [ ]:
import json
json.dump({"f1": 0.84, "json_valid": 0.99}, open("baseline.json", "w"))
json.dump({"f1": 0.81, "json_valid": 0.99}, open("eval_results.json", "w"))

In [ ]:
!python eval_gate.py; echo "mã thoát: $?"

f1           baseline 0.840 | mới 0.810 | TỤT
json_valid   baseline 0.990 | mới 0.990 | OK
mã thoát: 1


## Bài tập 5: Nhật ký thí nghiệm tối giản {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, time, uuid
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
open("runs.jsonl", "w").close()

def log_run(cfg, metrics):
    # TODO: ghi {"id", "time", "cfg", "metrics"} thành một dòng JSON
    pass

In [ ]:
def log_run(cfg, metrics):
    rec = {"id": uuid.uuid4().hex[:8], "time": time.strftime("%Y-%m-%d %H:%M:%S"), "cfg": cfg, "metrics": metrics}
    with open("runs.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")

for scale in (False, True):
    for C in (0.01, 1.0, 100.0):
        m = LogisticRegression(C=C, max_iter=5000)
        model = make_pipeline(StandardScaler(), m) if scale else m
        model.fit(X_tr, y_tr)
        log_run({"C": C, "scale": scale}, {"f1": f1_score(y_va, model.predict(X_va))})

df = pd.json_normalize([json.loads(l) for l in open("runs.jsonl")]).rename(columns=lambda c: c.split(".")[-1])
print(df[["id", "C", "scale", "f1"]].sort_values("f1", ascending=False).to_string(index=False))
print(df.groupby("scale")["f1"].mean().round(3))

      id      C  scale       f1
b453ec7e   1.00   True 0.967136
a4d00f77 100.00   True 0.966825
426e9934   0.01   True 0.954545
9a3d4d66   0.01  False 0.953271
ec90c229 100.00  False 0.952830
650cc219   1.00  False 0.943925
scale
False    0.950
True     0.963
Name: f1, dtype: float64


## Bài tập 6: Chạy lại ra đúng kết quả {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import random
import numpy as np, torch, torch.nn as nn

def dat_seed(seed):
    # TODO: cố định random, numpy, torch
    pass

def train_mot_lan():
    X = torch.tensor(np.random.normal(size=(256, 8)), dtype=torch.float32)
    y = (X[:, 0] > 0).long()
    model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    idx = list(range(256))
    for _ in range(50):
        random.shuffle(idx)
        b = torch.tensor(idx[:32])
        opt.zero_grad(); loss = nn.functional.cross_entropy(model(X[b]), y[b]); loss.backward(); opt.step()
    return loss.item()

In [ ]:
def dat_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

co_seed = []
for _ in range(2):
    dat_seed(42); co_seed.append(train_mot_lan())
khong_seed = [train_mot_lan(), train_mot_lan()]
print("cùng seed  :", co_seed)
print("không seed :", khong_seed)

cùng seed  : [0.3663865029811859, 0.3663865029811859]
không seed : [0.379963755607605, 0.2893966734409332]


## Bài tập 7: Phát hiện dữ liệu bị thay đổi bằng mã băm {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import hashlib, json, os
os.makedirs("data_v", exist_ok=True)
for ten, noi_dung in [("train.csv", "id,label\n1,0\n2,1\n"), ("val.csv", "id,label\n3,1\n"), ("test.csv", "id,label\n4,0\n")]:
    open(f"data_v/{ten}", "w").write(noi_dung)

def manifest(thu_muc):
    # TODO
    return {}

In [ ]:
def manifest(thu_muc):
    return {f: hashlib.sha256(open(os.path.join(thu_muc, f), "rb").read()).hexdigest()
            for f in sorted(os.listdir(thu_muc))}

v1 = manifest("data_v")
json.dump(v1, open("manifest_v1.json", "w"), indent=1)
open("data_v/train.csv", "w").write("id,label\n1,0\n2,0\n")          # sửa nhãn một dòng
open("data_v/extra.csv", "w").write("id,label\n5,1\n")
v2 = manifest("data_v")
thay_doi = [f for f in v1 if f in v2 and v1[f] != v2[f]]
them = [f for f in v2 if f not in v1]
xoa = [f for f in v1 if f not in v2]
print("thay đổi:", thay_doi, "| thêm:", them, "| xóa:", xoa)

thay đổi: ['train.csv'] | thêm: ['extra.csv'] | xóa: []
